# 11.26 — Goal-conditioned RL & hindsight replay

Goal-conditioned RL turns failed attempts into useful data for goals they did reach.

Goal-conditioned RL indexes the value table by desired outcome. Hindsight replay relabels a failed trajectory with the goal it actually achieved, increasing support for sparse rewards. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1125
rng = np.random.default_rng(SEED)
ACTIONS = np.array([0, 1, 2, 3])
DELTAS = {
    0: np.array([-1, 0]),
    1: np.array([0, 1]),
    2: np.array([1, 0]),
    3: np.array([0, -1]),
}
ACTION_NAMES = np.array(["up", "right", "down", "left"])


def discounted_return(rewards, gamma):
    total = 0.0
    for t, reward in enumerate(rewards):
        total += (gamma ** t) * reward
    return float(total)


def td_update(q_old, reward, next_value, alpha, gamma):
    target = reward + gamma * next_value
    q_new = q_old + alpha * (target - q_old)
    return float(target), float(q_new)


def make_grid_env(name, width, height, start, goals, walls=None, slip=0.0, wind=None, step_cost=-0.02, max_steps=40):
    walls = set(walls or [])
    goals = dict(goals)
    wind = wind or {}
    states = []
    index = {}
    for row in range(height):
        for col in range(width):
            pos = (row, col)
            if pos not in walls:
                index[pos] = len(states)
                states.append(pos)
    env = {
        "name": name,
        "width": width,
        "height": height,
        "start": start,
        "goals": goals,
        "walls": walls,
        "slip": slip,
        "wind": wind,
        "step_cost": step_cost,
        "max_steps": max_steps,
        "states": states,
        "index": index,
    }
    return env


def move_position(env, pos, action):
    drift = env["wind"].get(pos, np.array([0, 0]))
    raw = np.array(pos) + DELTAS[int(action)] + drift
    row = int(np.clip(raw[0], 0, env["height"] - 1))
    col = int(np.clip(raw[1], 0, env["width"] - 1))
    candidate = (row, col)
    if candidate in env["walls"]:
        return pos
    return candidate


def sample_step(env, pos, action, local_rng):
    chosen = int(action)
    if local_rng.random() < env["slip"]:
        chosen = int(local_rng.choice(ACTIONS))
    next_pos = move_position(env, pos, chosen)
    reward = env["step_cost"]
    done = False
    if next_pos in env["goals"]:
        reward += env["goals"][next_pos]
        done = True
    return next_pos, float(reward), done


def greedy_action(q_values, state_index, allowed=None):
    values = q_values[state_index].copy()
    if allowed is not None:
        values = np.where(allowed[state_index], values, -1e9)
    return int(np.argmax(values))


def rollout_return(env, q_values, episodes=20, gamma=0.9, support=None, seed=0, goal=None):
    local_rng = np.random.default_rng(seed)
    returns = []
    successes = []
    for episode in range(episodes):
        pos = env["start"]
        total = 0.0
        discount = 1.0
        success = 0.0
        for step in range(env["max_steps"]):
            state = env["index"][pos]
            action = greedy_action(q_values, state, support)
            pos, reward, done = sample_step(env, pos, action, local_rng)
            if goal is not None:
                reward = 1.0 if pos == goal else -0.02
                done = pos == goal
            total += discount * reward
            discount *= gamma
            if done:
                success = 1.0
                break
        returns.append(total)
        successes.append(success)
    return float(np.mean(returns)), float(np.mean(successes))


def q_learning(env, episodes=120, alpha=0.35, gamma=0.9, epsilon=0.2, shaping=None, goal=None, seed=0):
    local_rng = np.random.default_rng(seed)
    q_values = np.zeros((len(env["states"]), len(ACTIONS)))
    history = []
    for episode in range(episodes):
        pos = env["start"]
        total = 0.0
        discount = 1.0
        for step in range(env["max_steps"]):
            state = env["index"][pos]
            if local_rng.random() < epsilon:
                action = int(local_rng.choice(ACTIONS))
            else:
                action = int(np.argmax(q_values[state]))
            next_pos, reward, done = sample_step(env, pos, action, local_rng)
            if goal is not None:
                reward = 1.0 if next_pos == goal else -0.02
                done = next_pos == goal
            shaped_reward = reward
            if shaping is not None:
                shaped_reward += shaping(pos, next_pos)
            next_state = env["index"][next_pos]
            target = shaped_reward + gamma * np.max(q_values[next_state]) * (1.0 - float(done))
            q_values[state, action] += alpha * (target - q_values[state, action])
            total += discount * reward
            discount *= gamma
            pos = next_pos
            if done:
                break
        history.append(total)
    return q_values, np.array(history)


def value_iteration(env, gamma=0.9, reward_bonus=None, iterations=80):
    values = np.zeros(len(env["states"]))
    q_values = np.zeros((len(env["states"]), len(ACTIONS)))
    for iteration in range(iterations):
        new_values = values.copy()
        for pos in env["states"]:
            state = env["index"][pos]
            candidates = []
            for action in ACTIONS:
                next_pos = move_position(env, pos, int(action))
                reward = env["step_cost"] + env["goals"].get(next_pos, 0.0)
                if reward_bonus is not None:
                    reward += reward_bonus(pos, next_pos)
                done = next_pos in env["goals"]
                next_state = env["index"][next_pos]
                candidates.append(reward + gamma * values[next_state] * (1.0 - float(done)))
            q_values[state] = np.array(candidates)
            new_values[state] = float(np.max(candidates))
        values = new_values
    return values, q_values


def potential_to_goal(env, goal):
    distances = {}
    for pos in env["states"]:
        distances[pos] = abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])
    max_distance = max(distances.values()) + 1.0
    return {pos: -dist / max_distance for pos, dist in distances.items()}


def rl_ladder(goal_conditioned=False):
    rungs = []
    rungs.append(make_grid_env("D1 2-state chain", 2, 1, (0, 0), {(0, 1): 1.0}, max_steps=4))
    rungs.append(make_grid_env("D2 slippery 3-state chain", 3, 1, (0, 0), {(0, 2): 1.0}, slip=0.15, max_steps=8))
    rungs.append(make_grid_env("D3 4x4 gridworld", 4, 4, (3, 0), {(0, 3): 1.0}, walls={(1, 1), (2, 1)}, max_steps=28))
    rungs.append(make_grid_env("D4 stochastic windy grid", 5, 5, (4, 0), {(0, 4): 1.0}, walls={(1, 1), (2, 1), (3, 3)}, slip=0.10, wind={(3, 2): np.array([-1, 0]), (2, 3): np.array([-1, 0])}, max_steps=40))
    goals = {(0, 5): 1.0}
    if goal_conditioned:
        goals[(5, 0)] = 0.8
        goals[(2, 5)] = 0.7
    rungs.append(make_grid_env("D5 larger sparse-reward grid", 6, 6, (5, 0), goals, walls={(1, 1), (1, 2), (2, 2), (3, 4), (4, 1)}, slip=0.12, wind={(4, 3): np.array([-1, 0]), (3, 3): np.array([-1, 0])}, max_steps=55))
    return rungs


def plot_policy(ax, env, q_values, title):
    grid = np.full((env["height"], env["width"]), np.nan)
    arrows = ["↑", "→", "↓", "←"]
    for pos in env["states"]:
        state = env["index"][pos]
        grid[pos] = np.max(q_values[state])
    ax.imshow(grid, cmap="viridis")
    for pos in env["states"]:
        state = env["index"][pos]
        label = "G" if pos in env["goals"] else arrows[int(np.argmax(q_values[state]))]
        ax.text(pos[1], pos[0], label, ha="center", va="center", color="white")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

The lesson's common discounted-consequence calculation is the guardrail: rewards `[1, 0, 2]` with $\gamma=0.9$ give $G=2.620$, and the one-step target $1+0.9\cdot0.8$ gives $1.720$. This notebook then specializes that backup for Goal-conditioned RL & hindsight replay.

$$Q(s,a,g)\leftarrow Q(s,a,g)+\alpha\bigl(r_g+\gamma\max_{a'}Q(s',a',g)-Q(s,a,g)\bigr)$$

In [ ]:

rewards = [1.0, 0.0, 2.0]
gamma = 0.9
three_step_return = discounted_return(rewards, gamma)
target, q_new = td_update(q_old=0.4, reward=1.0, next_value=0.8, alpha=0.5, gamma=gamma)

print("three-step return", round(three_step_return, 3))
print("one-step target", round(target, 3))
print("updated Q", round(q_new, 3))

assert round(three_step_return, 3) == 2.620
assert round(target, 3) == 1.720
assert round(q_new, 3) == 1.060


Now define `goal_q_with_her` once on D1. The assert is intentionally small and exact, so the same method can scale to D2-D5 without changing the objective or metric.

In [ ]:

def goal_q_with_her(env, trajectory, desired_goal, gamma=0.9, alpha=0.5):
    goals = list(env["goals"].keys())
    if desired_goal not in goals:
        goals.append(desired_goal)
    goal_index = {goal: i for i, goal in enumerate(goals)}
    q_values = np.zeros((len(env["states"]), len(ACTIONS), len(goals)))
    updates = []
    achieved_goal = trajectory[-1][2]
    relabel_goals = [desired_goal, achieved_goal]
    for relabel_goal in relabel_goals:
        g_idx = goal_index[relabel_goal]
        for pos, action, next_pos in trajectory:
            state = env["index"][pos]
            next_state = env["index"][next_pos]
            reward = 1.0 if next_pos == relabel_goal else 0.0
            target = reward + gamma * np.max(q_values[next_state, :, g_idx])
            q_values[state, action, g_idx] += alpha * (target - q_values[state, action, g_idx])
            updates.append((relabel_goal, reward))
    return q_values, updates, goal_index


d1 = make_grid_env("D1 2-state two-goal chain", 2, 1, (0, 0), {(0, 1): 1.0, (0, 0): 0.5}, max_steps=4)
trajectory = [((0, 0), 1, (0, 1))]
q_values, updates, goal_index = goal_q_with_her(d1, trajectory, desired_goal=(0, 0))
start_state = d1["index"][(0, 0)]
achieved_idx = goal_index[(0, 1)]
desired_idx = goal_index[(0, 0)]
print("updates", updates)
print("hindsight value", q_values[start_state, 1, achieved_idx])

assert len(goal_index) == 2
assert q_values[start_state, 1, achieved_idx] == 0.5
assert q_values[start_state, 1, desired_idx] == 0.0


## The dataset ladder

Family F12 is built inline: D1 is inspectable, D2 adds stochasticity, D3 is a 4x4 grid, D4 adds wind/slip, and D5 is a larger sparse-reward grid. The preview reports sizes before any learning code is used.

In [ ]:

rungs = rl_ladder(goal_conditioned=True)
for name, env in [(env["name"], env) for env in rungs]:
    size = len(env["states"])
    goal_count = len(env["goals"])
    print(f"{name}: states={size}, actions={len(ACTIONS)}, goals={goal_count}, slip={env['slip']}")
    print("  start", env["start"], "sample states", env["states"][:4])


## Run the same method across D1-D5

The code below keeps one algorithmic idea fixed and reports the plan metric: goal success return. Seeds are fixed and all runs are CPU-only.

In [ ]:

def train_goal_q_with_her(env, desired_goal, episodes=120, alpha=0.35, gamma=0.9, epsilon=0.2, use_her=True, seed=0):
    local_rng = np.random.default_rng(seed)
    goals = list(env["goals"].keys())
    if desired_goal not in goals:
        goals.append(desired_goal)
    goal_index = {goal: idx for idx, goal in enumerate(goals)}
    q_values = np.zeros((len(env["states"]), len(ACTIONS), len(goals)))
    history = []
    for episode in range(episodes):
        pos = env["start"]
        trajectory = []
        total = 0.0
        discount = 1.0
        g_idx = goal_index[desired_goal]
        for step in range(env["max_steps"]):
            state = env["index"][pos]
            if local_rng.random() < epsilon:
                action = int(local_rng.choice(ACTIONS))
            else:
                action = int(np.argmax(q_values[state, :, g_idx]))
            next_pos, base_reward, done = sample_step(env, pos, action, local_rng)
            reward = 1.0 if next_pos == desired_goal else -0.02
            done = next_pos == desired_goal
            trajectory.append((pos, action, next_pos))
            next_state = env["index"][next_pos]
            target = reward + gamma * np.max(q_values[next_state, :, g_idx]) * (1.0 - float(done))
            q_values[state, action, g_idx] += alpha * (target - q_values[state, action, g_idx])
            total += discount * reward
            discount *= gamma
            pos = next_pos
            if done:
                break
        if use_her and trajectory:
            achieved_goal = trajectory[-1][2]
            if achieved_goal not in goal_index:
                goal_index[achieved_goal] = len(goals)
                goals.append(achieved_goal)
                extra = np.zeros((len(env["states"]), len(ACTIONS), 1))
                q_values = np.concatenate([q_values, extra], axis=2)
            her_idx = goal_index[achieved_goal]
            for pos, action, next_pos in trajectory:
                state = env["index"][pos]
                next_state = env["index"][next_pos]
                reward = 1.0 if next_pos == achieved_goal else -0.02
                done = next_pos == achieved_goal
                target = reward + gamma * np.max(q_values[next_state, :, her_idx]) * (1.0 - float(done))
                q_values[state, action, her_idx] += alpha * (target - q_values[state, action, her_idx])
        history.append(total)
    return q_values, goal_index, np.array(history)


rungs = rl_ladder(goal_conditioned=True)
rows = []
artifacts = []
for rung_id, env in enumerate(rungs, start=1):
    goal = max(env["goals"], key=env["goals"].get)
    plain_all_q, plain_goal_index, plain_history = train_goal_q_with_her(env, goal, use_her=False, seed=300 + rung_id)
    her_all_q, her_goal_index, her_history = train_goal_q_with_her(env, goal, use_her=True, seed=500 + rung_id)
    plain_q = plain_all_q[:, :, plain_goal_index[goal]]
    her_q = her_all_q[:, :, her_goal_index[goal]]
    plain_return, plain_success = rollout_return(env, plain_q, goal=goal, seed=600 + rung_id)
    her_return, her_success = rollout_return(env, her_q, goal=goal, seed=700 + rung_id)
    rows.append((env["name"], plain_success, her_success, her_return))
    artifacts.append((env, her_q, her_history))

print("rung | no HER success | HER success | HER return")
for name, plain_success, her_success, her_return in rows:
    print(f"{name} | {plain_success:.2f} | {her_success:.2f} | {her_return:.3f}")


## Results visualization

The closing figure has two parts: top-row small multiples show the learned artifact for each rung; the bottom-left summary curve plots the metric from D1 to D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, artifact in zip(axes[0], artifacts):
    env, q_values, history = artifact
    plot_policy(ax, env, q_values, env["name"].split()[0])
metric_values = []
for artifact in artifacts:
    history = artifact[2]
    metric_values.append(float(np.mean(history[-10:])))
axes[1, 0].plot(range(1, len(metric_values) + 1), metric_values, marker="o")
axes[1, 0].set_title("goal success return vs rung")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("goal success return")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5

Named pitfall: ignoring policy support leaves unreached goals with no useful backup targets. The first line reproduces the wrong behavior; the fix keeps the lesson's discounted objective and improves the reported behavior.

In [ ]:

d5 = rl_ladder(goal_conditioned=True)[4]
unreached_goal = (2, 5)
plain_all_q, plain_goal_index, plain_history = train_goal_q_with_her(d5, unreached_goal, episodes=40, use_her=False, seed=1600)
her_all_q, her_goal_index, her_history = train_goal_q_with_her(d5, unreached_goal, episodes=40, use_her=True, seed=1700)
plain_q = plain_all_q[:, :, plain_goal_index[unreached_goal]]
her_q = her_all_q[:, :, her_goal_index[unreached_goal]]
plain_return, plain_success = rollout_return(d5, plain_q, goal=unreached_goal, seed=1800)
her_return, her_success = rollout_return(d5, her_q, goal=unreached_goal, seed=1900)
print("without relabel support success", plain_success)
print("with hindsight relabel success", her_success)


## Evaluate it + Practice

- Metric: track goal success return against a no-skill random or unsupported-action baseline.
- Sanity check: D1 must match the exact asserts above before trusting D5.
- Ablation: turn off the key idea (`goal_q_with_her`) and the metric should drop or become less stable.
- Failure signal: values improve while realized return, regret, or success rate does not.
- CPU rule: keep seeds fixed and do not scale episodes until the small ladder behaves.

Practice prompts:
1. Change $\gamma$ from 0.9 to 0.7 and predict which rung changes most.
2. Add one wall to D3 and rerun the same method without changing its API.
3. Replace the no-skill baseline with a hand-coded greedy baseline and compare.